# 📚 Unity Catalog & Access Control - Complete Guide

This notebook covers three critical Databricks concepts:

## Topics Covered

1. **Unity Catalog** - Unified governance solution for data and AI
2. **Access Control** - Fine-grained permissions and security
3. **Control Plane vs. Data Plane** - Databricks architecture fundamentals

---

## Navigation

* [Unity Catalog Overview](#cell-2)
* [Unity Catalog Architecture](#cell-3)
* [Unity Catalog Code Examples](#cell-4)
* [Access Control Overview](#cell-7)
* [Granting Permissions](#cell-8)
* [Control Plane vs Data Plane](#cell-11)

---

**Prerequisites**: Basic SQL and Python knowledge

**Compute**: Works with any Databricks cluster or SQL Warehouse

## 🏛️ Unity Catalog Overview

**Unity Catalog** is Databricks' unified governance solution for data and AI assets across clouds.

### Key Features

* **Three-level namespace**: `catalog.schema.table`
* **Unified governance**: Manage data, ML models, notebooks, files, and more
* **Fine-grained access control**: Column-level, row-level, and table-level permissions
* **Data lineage**: Track data flow across tables and notebooks
* **Audit logging**: Track all access and changes
* **Data sharing**: Securely share data with Delta Sharing
* **PII protection**: Built-in column masking and row filtering

### Three-Level Namespace

```
catalog               (e.g., production, dev, analytics)
  └── schema          (e.g., sales, customers, inventory)
       └── table      (e.g., orders, users, products)
```

### Benefits

* **Centralized governance** across all workspaces
* **Consistent security** policies
* **Cross-workspace** data sharing
* **Automated lineage** tracking
* **Compliance** with regulations (GDPR, HIPAA, etc.)

## 🏭 Unity Catalog Architecture

### Metastore

The **metastore** is the top-level container for metadata in Unity Catalog.

* One metastore per region
* Can be attached to multiple workspaces
* Contains all catalogs, schemas, and tables
* Stores metadata in cloud storage (S3, ADLS, GCS)

### Hierarchy

```
Metastore (region-level)
  ├── Catalog: production
  │    ├── Schema: sales
  │    │    ├── Table: orders
  │    │    └── Table: customers
  │    └── Schema: marketing
  │         └── Table: campaigns
  ├── Catalog: dev
  │    └── Schema: experiments
  └── Catalog: analytics
       └── Schema: reporting
```

### Security Model

* **Account admins**: Manage metastore and account-level settings
* **Metastore admins**: Manage catalogs and top-level permissions
* **Catalog owners**: Manage schemas within their catalog
* **Schema owners**: Manage tables within their schema
* **Grant-based access**: Explicit permissions required for data access

## 💻 Unity Catalog Code Examples

Let's create Unity Catalog objects and see how to work with them.

In [0]:
%sql
-- Create a new catalog
CREATE CATALOG IF NOT EXISTS my_catalog
COMMENT 'Demo catalog for Unity Catalog examples';

-- Use the catalog
USE CATALOG my_catalog;

-- Create schemas
CREATE SCHEMA IF NOT EXISTS sales
COMMENT 'Sales data and customer information';

CREATE SCHEMA IF NOT EXISTS analytics
COMMENT 'Analytics and reporting tables';

-- Show all catalogs
SHOW CATALOGS;

In [0]:
%sql
-- Create a customers table with PII data
USE CATALOG my_catalog;
USE SCHEMA sales;

CREATE OR REPLACE TABLE customers (
  customer_id BIGINT COMMENT 'Unique customer identifier',
  email STRING COMMENT 'Email address (PII)',
  phone STRING COMMENT 'Phone number (PII)',
  ssn STRING COMMENT 'Social Security Number (PII - highly sensitive)',
  first_name STRING COMMENT 'First name (PII)',
  last_name STRING COMMENT 'Last name (PII)',
  address STRING COMMENT 'Street address (PII)',
  city STRING,
  state STRING,
  zip_code STRING,
  date_of_birth DATE COMMENT 'Date of birth (PII)',
  credit_score INT,
  account_balance DECIMAL(10,2),
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
COMMENT 'Customer data with PII fields';

-- Insert sample data
INSERT INTO customers VALUES
  (1, 'john.doe@email.com', '555-0101', '123-45-6789', 'John', 'Doe', '123 Main St', 'Seattle', 'WA', '98101', '1985-03-15', 720, 15000.50, current_timestamp(), current_timestamp()),
  (2, 'jane.smith@email.com', '555-0102', '987-65-4321', 'Jane', 'Smith', '456 Oak Ave', 'Portland', 'OR', '97201', '1990-07-22', 780, 25000.00, current_timestamp(), current_timestamp()),
  (3, 'bob.johnson@email.com', '555-0103', '456-78-9012', 'Bob', 'Johnson', '789 Pine Rd', 'San Francisco', 'CA', '94102', '1982-11-30', 650, 8500.75, current_timestamp(), current_timestamp());

SELECT * FROM customers;

## 🔒 PII Protection - Column Masking

**Column masking** allows you to protect sensitive data by transforming values based on user permissions.

### Masking Functions

* **MASK()**: Replaces characters with 'X' or other mask character
* **MASK_HASH()**: Returns hash of the value
* **Custom functions**: Create your own masking logic

### Use Cases

* Mask SSN: `XXX-XX-6789` (show last 4 digits)
* Mask email: `j***@email.com`
* Mask phone: `555-****`
* Hash sensitive fields for analytics teams

In [0]:
%sql
-- Create a mask function to show only last 4 digits of SSN
CREATE OR REPLACE FUNCTION my_catalog.sales.mask_ssn(ssn STRING)
RETURNS STRING
RETURN CONCAT('XXX-XX-', RIGHT(ssn, 4));

-- Create a mask function for email
CREATE OR REPLACE FUNCTION my_catalog.sales.mask_email(email STRING)
RETURNS STRING
RETURN CONCAT(LEFT(email, 1), '***@', SPLIT(email, '@')[1]);

-- Create a mask function for phone
CREATE OR REPLACE FUNCTION my_catalog.sales.mask_phone(phone STRING)
RETURNS STRING
RETURN CONCAT(LEFT(phone, 4), '****');

-- Apply column masks to the customers table
ALTER TABLE my_catalog.sales.customers 
ALTER COLUMN ssn SET MASK my_catalog.sales.mask_ssn;

ALTER TABLE my_catalog.sales.customers 
ALTER COLUMN email SET MASK my_catalog.sales.mask_email;

ALTER TABLE my_catalog.sales.customers 
ALTER COLUMN phone SET MASK my_catalog.sales.mask_phone;

-- Now when users without UNMASK privilege query, they see masked data
SELECT customer_id, email, phone, ssn, first_name, last_name 
FROM my_catalog.sales.customers;

## 🔐 Access Control Overview

**Access control** in Unity Catalog provides fine-grained permissions at multiple levels.

### Permission Levels

1. **Catalog-level**: Controls access to entire catalog
2. **Schema-level**: Controls access to schemas within a catalog
3. **Table-level**: Controls access to specific tables
4. **Column-level**: Controls access to specific columns (via masking)
5. **Row-level**: Controls access to specific rows (via row filters)

### Common Privileges

* **USE CATALOG**: Required to access a catalog
* **USE SCHEMA**: Required to access a schema
* **SELECT**: Read data from tables
* **INSERT**: Add new rows
* **UPDATE**: Modify existing rows
* **DELETE**: Remove rows
* **MODIFY**: Change table structure
* **READ FILES**: Read underlying data files
* **WRITE FILES**: Write to underlying data files
* **CREATE**: Create new objects (schemas, tables, etc.)
* **OWNER**: Full control over an object

## ✅ Granting Permissions - Examples

Let's see how to grant different types of permissions to users and groups.

In [0]:
%sql
-- First, let's check who has access to the catalog already
SHOW GRANTS ON CATALOG my_catalog;

-- Grant USE CATALOG to yourself (replace with your actual email)
GRANT USE CATALOG ON CATALOG my_catalog TO `aralokesh@gmail.com`;

-- Grant USE SCHEMA to yourself
GRANT USE SCHEMA ON SCHEMA my_catalog.sales TO `aralokesh@gmail.com`;

-- Grant SELECT on all tables in a schema
GRANT SELECT ON SCHEMA my_catalog.sales TO `aralokesh@gmail.com`;

-- To grant to a GROUP (if you have groups set up):
-- GRANT USE SCHEMA ON SCHEMA my_catalog.sales TO `your_group_name`;

-- Grant CREATE privileges (optional)
GRANT CREATE TABLE ON SCHEMA my_catalog.sales TO `aralokesh@gmail.com`;
GRANT CREATE SCHEMA ON CATALOG my_catalog TO `aralokesh@gmail.com`;

-- Verify grants
SHOW GRANTS ON CATALOG my_catalog;
SHOW GRANTS ON SCHEMA my_catalog.sales;

In [0]:
%sql
-- Grant SELECT on specific table to yourself
GRANT SELECT ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- Grant INSERT and UPDATE to yourself
GRANT INSERT, UPDATE ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- Grant MODIFY (allows schema changes)
GRANT MODIFY ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- For production: Grant to specific groups instead
-- GRANT SELECT ON TABLE my_catalog.sales.customers TO `data_analysts`;
-- GRANT INSERT, UPDATE ON TABLE my_catalog.sales.customers TO `data_engineers`;

-- Show all grants on the table
SHOW GRANTS ON TABLE my_catalog.sales.customers;

In [0]:
%sql
-- By default, you'll see MASKED data due to the column masks we set earlier
-- To see unmasked PII data, you need UNMASK privilege

-- Grant yourself UNMASK privilege to see unmasked data
GRANT SELECT, UNMASK ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- For production: Grant UNMASK only to authorized roles
-- GRANT SELECT, UNMASK ON TABLE my_catalog.sales.customers TO `compliance_team`;

-- Grant UNMASK for specific columns only
-- GRANT UNMASK ON TABLE my_catalog.sales.customers 
--   COLUMNS (ssn, email) 
--   TO `security_team`;

-- To revoke UNMASK:
-- REVOKE UNMASK ON TABLE my_catalog.sales.customers FROM `aralokesh@gmail.com`;

-- Query to see if data is masked or unmasked
SELECT customer_id, email, phone, ssn, first_name, last_name
FROM my_catalog.sales.customers;

## 🛡️ Row-Level Security

**Row filters** allow you to restrict which rows users can see based on conditions.

### Use Cases

* Regional managers see only their region's data
* Sales reps see only their own customers
* Multi-tenant applications with data isolation
* Compliance requirements (e.g., EU users see only EU data)

In [0]:
%sql
-- Create a row filter function that filters by state
CREATE OR REPLACE FUNCTION my_catalog.sales.filter_by_state(state STRING)
RETURNS BOOLEAN
RETURN 
  CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('west_coast_team') THEN state IN ('CA', 'OR', 'WA')
    WHEN IS_ACCOUNT_GROUP_MEMBER('east_coast_team') THEN state IN ('NY', 'MA', 'PA')
    WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN TRUE
    ELSE FALSE
  END;

-- Apply row filter to customers table
ALTER TABLE my_catalog.sales.customers
SET ROW FILTER my_catalog.sales.filter_by_state ON (state);

-- Now users will only see rows they're authorized to see
SELECT customer_id, first_name, last_name, city, state
FROM my_catalog.sales.customers;

-- To remove row filter:
-- ALTER TABLE my_catalog.sales.customers DROP ROW FILTER;

## 🔄 CDC (Change Data Capture) Overview

**Change Data Capture (CDC)** is a design pattern that tracks changes in source data and propagates them to target systems.

### What is CDC?

CDC captures:
* **INSERT**: New rows added
* **UPDATE**: Existing rows modified
* **DELETE**: Rows removed

Instead of full data loads, CDC only processes **changed data**, making pipelines more efficient.

### Why Use CDC?

* **Efficiency**: Process only changes, not full datasets
* **Real-time**: Near real-time data synchronization
* **History tracking**: Maintain complete audit trail
* **Reduced load**: Less compute and storage for incremental updates
* **Data freshness**: Keep downstream systems up-to-date

### CDC Use Cases

* Sync operational databases to data warehouses
* Build Type 2 Slowly Changing Dimensions (SCD Type 2)
* Create audit logs and data lineage
* Real-time analytics dashboards
* Data replication across systems

## 🏛️ CDC Architecture in Databricks

### CDC Approaches in Databricks

1. **Delta Lake MERGE**: Manual CDC with MERGE operations
2. **Auto Loader CDC**: Streaming CDC from cloud storage
3. **Lakeflow Pipelines CDC**: Declarative CDC with APPLY CHANGES INTO
4. **Debezium/Kafka**: External CDC tools streaming to Databricks

### Delta Lake Change Data Feed

Delta Lake tracks all changes with **Change Data Feed (CDF)**:

```
ENABLE CHANGE DATA FEED on table
↓
Delta automatically tracks:
- _change_type: insert, update_preimage, update_postimage, delete
- _commit_version: Version number
- _commit_timestamp: When change occurred
```

### CDC Pipeline Flow

```
Source System (e.g., MySQL)
  ↓
CDC Capture (Debezium/Binary Logs)
  ↓
Message Queue (Kafka/Kinesis)
  ↓
Auto Loader (Databricks)
  ↓
Bronze Table (Raw CDC events)
  ↓
Silver Table (Merged/Applied changes)
  ↓
Gold Table (Business logic applied)
```

## 💻 CDC Code Examples

Let's implement CDC patterns in Databricks.

In [0]:
%sql
-- Enable Change Data Feed on existing table
ALTER TABLE my_catalog.sales.customers 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Create new table with CDF enabled
CREATE TABLE my_catalog.sales.orders (
  order_id BIGINT,
  customer_id BIGINT,
  order_date DATE,
  total_amount DECIMAL(10,2),
  status STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Insert some test data
INSERT INTO my_catalog.sales.orders VALUES
  (1001, 1, '2024-01-15', 150.00, 'completed'),
  (1002, 2, '2024-01-16', 275.50, 'pending'),
  (1003, 3, '2024-01-17', 89.99, 'completed');

SELECT * FROM my_catalog.sales.orders;

In [0]:
%sql
-- Read all changes since version 0
SELECT *
FROM table_changes('my_catalog.sales.orders', 0);

-- Make some changes to generate CDC events
UPDATE my_catalog.sales.orders 
SET status = 'shipped' 
WHERE order_id = 1002;

INSERT INTO my_catalog.sales.orders VALUES
  (1004, 1, '2024-01-18', 320.00, 'pending');

DELETE FROM my_catalog.sales.orders 
WHERE order_id = 1003;

-- Now read changes again to see CDC events
SELECT 
  order_id,
  customer_id,
  status,
  _change_type,
  _commit_version,
  _commit_timestamp
FROM table_changes('my_catalog.sales.orders', 1)
ORDER BY _commit_version, _commit_timestamp;

In [0]:
# Create a source table with changes
source_changes = spark.createDataFrame([
    (1001, 1, '2024-01-15', 150.00, 'delivered', 'UPDATE'),
    (1002, 2, '2024-01-16', 275.50, 'shipped', 'UPDATE'),
    (1005, 2, '2024-01-19', 450.00, 'pending', 'INSERT'),
    (1004, 1, '2024-01-18', 320.00, 'cancelled', 'UPDATE')
], ['order_id', 'customer_id', 'order_date', 'total_amount', 'status', 'operation'])

# Convert date string to date type
from pyspark.sql.functions import col, to_date, current_timestamp

source_changes = source_changes.withColumn('order_date', to_date(col('order_date')))

# Apply CDC changes using MERGE
source_changes.createOrReplaceTempView('source_changes')

spark.sql("""
  MERGE INTO my_catalog.sales.orders AS target
  USING source_changes AS source
  ON target.order_id = source.order_id
  WHEN MATCHED AND source.operation = 'UPDATE' THEN
    UPDATE SET
      target.status = source.status,
      target.total_amount = source.total_amount
  WHEN MATCHED AND source.operation = 'DELETE' THEN
    DELETE
  WHEN NOT MATCHED AND source.operation = 'INSERT' THEN
    INSERT (order_id, customer_id, order_date, total_amount, status)
    VALUES (source.order_id, source.customer_id, source.order_date, source.total_amount, source.status)
""")

print("CDC changes applied successfully!")

# Verify the results
display(spark.sql("SELECT * FROM my_catalog.sales.orders ORDER BY order_id"))

In [0]:
# Create SCD Type 2 table for customer history
spark.sql("""
  CREATE OR REPLACE TABLE my_catalog.sales.customers_history (
    customer_id BIGINT,
    email STRING,
    phone STRING,
    first_name STRING,
    last_name STRING,
    city STRING,
    state STRING,
    credit_score INT,
    account_balance DECIMAL(10,2),
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
  )
  TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# Initial load - mark all records as current
spark.sql("""
  INSERT INTO my_catalog.sales.customers_history
  SELECT 
    customer_id,
    email,
    phone,
    first_name,
    last_name,
    city,
    state,
    credit_score,
    account_balance,
    created_at as start_date,
    NULL as end_date,
    true as is_current
  FROM my_catalog.sales.customers
""")

print("SCD Type 2 table initialized")

# Simulate a change - customer moved to a new city
spark.sql("""
  UPDATE my_catalog.sales.customers
  SET city = 'Austin', state = 'TX', updated_at = current_timestamp()
  WHERE customer_id = 1
""")

print("Customer data updated - now let's apply SCD Type 2 logic")

In [0]:
# Read changes from source table
changes_df = spark.sql("""
  SELECT 
    customer_id,
    email,
    phone,
    first_name,
    last_name,
    city,
    state,
    credit_score,
    account_balance,
    updated_at
  FROM my_catalog.sales.customers
  WHERE updated_at > (SELECT MAX(start_date) FROM my_catalog.sales.customers_history)
""")

if changes_df.count() > 0:
    changes_df.createOrReplaceTempView('customer_changes')
    
    # Step 1: Close out old records (set end_date and is_current = false)
    spark.sql("""
      MERGE INTO my_catalog.sales.customers_history AS target
      USING customer_changes AS source
      ON target.customer_id = source.customer_id 
         AND target.is_current = true
      WHEN MATCHED THEN
        UPDATE SET
          target.end_date = source.updated_at,
          target.is_current = false
    """)
    
    # Step 2: Insert new records
    spark.sql("""
      INSERT INTO my_catalog.sales.customers_history
      SELECT 
        customer_id,
        email,
        phone,
        first_name,
        last_name,
        city,
        state,
        credit_score,
        account_balance,
        updated_at as start_date,
        NULL as end_date,
        true as is_current
      FROM customer_changes
    """)
    
    print(f"Applied SCD Type 2 changes for {changes_df.count()} customers")
else:
    print("No changes detected")

# View customer history - see the change over time
display(spark.sql("""
  SELECT 
    customer_id,
    first_name,
    last_name,
    city,
    state,
    start_date,
    end_date,
    is_current
  FROM my_catalog.sales.customers_history
  WHERE customer_id = 1
  ORDER BY start_date
"""))

## 🌊 CDC with Lakeflow Pipelines

**Lakeflow Spark Declarative Pipelines** (formerly DLT) provide declarative CDC with `APPLY CHANGES INTO`.

### Benefits

* Declarative syntax - no manual MERGE logic
* Automatic SCD Type 1 or Type 2
* Built-in data quality checks
* Automatic retry and error handling
* Lineage tracking

### Example Pipeline Code

```python
import dlt
from pyspark.sql.functions import col, expr

# Bronze layer - raw CDC events
@dlt.table(
  comment="Raw CDC events from source system",
  table_properties={"quality": "bronze"}
)
def orders_cdc_bronze():
  return (
    spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .load("/mnt/source/cdc/orders/")
  )

# Silver layer - apply CDC changes
dlt.create_streaming_table("orders_silver")

dlt.apply_changes(
  target = "orders_silver",
  source = "orders_cdc_bronze",
  keys = ["order_id"],
  sequence_by = col("updated_timestamp"),
  stored_as_scd_type = 1,  # Use 2 for SCD Type 2
  except_column_list = ["operation", "_metadata"]
)
```

### Key Parameters

* **target**: Destination table name
* **source**: Source table or query
* **keys**: Primary key columns
* **sequence_by**: Column to determine order of changes
* **stored_as_scd_type**: 1 for current state, 2 for history
* **except_column_list**: Columns to exclude from target

## ✈️ Control Plane vs Data Plane

**Databricks architecture** is split into two main components:

### Control Plane

The **control plane** manages workspace configuration and metadata.

**Located in**: Databricks-managed cloud account

**Responsibilities**:
* Workspace UI and notebooks
* Cluster management (create, start, stop, delete)
* Job scheduling and orchestration
* Notebook and code versioning
* Access control and user management
* Audit logs and monitoring
* Unity Catalog metastore

**Components**:
* Web application (workspace UI)
* REST APIs
* Job scheduler
* Cluster manager
* Authentication service

### Data Plane

The **data plane** processes and stores actual data.

**Located in**: Customer's cloud account (your AWS/Azure/GCP)

**Responsibilities**:
* Compute clusters (VMs/instances)
* Data processing and query execution
* Data storage (Delta Lake tables)
* Network connections to data sources
* Actual workload execution

**Components**:
* Spark clusters
* Storage (S3, ADLS, GCS)
* Delta Lake
* ML model training infrastructure

## 🏛️ Architecture Diagram

```
┌────────────────────────────────────────────────┐
│         CONTROL PLANE (Databricks Cloud)            │
│────────────────────────────────────────────────│
│                                                  │
│  💻 Workspace UI    📝 Notebooks              │
│  🛠️  Cluster Manager  📅 Job Scheduler          │
│  🔐 Access Control   📊 Monitoring              │
│  🏛️  Unity Catalog    🔍 Audit Logs              │
│                                                  │
└──────────────────────┬─────────────────────────┘
                       │
                       │ Secure API calls
                       │
┌──────────────────────┴─────────────────────────┐
│        DATA PLANE (Your Cloud Account)             │
│────────────────────────────────────────────────│
│                                                  │
│  ⚙️  Spark Clusters (EC2/VMs)                   │
│  💾 Data Storage (S3/ADLS/GCS)                 │
│  📦 Delta Lake Tables                          │
│  🌐 Network & VPC Configuration                │
│  🔒 Data Encryption (at rest & in transit)     │
│                                                  │
└────────────────────────────────────────────────┘
```

### Key Benefits

* **Security**: Your data never leaves your cloud account
* **Compliance**: Data residency requirements easily met
* **Performance**: Data processing happens close to storage
* **Scalability**: Databricks manages control plane; you scale data plane
* **Cost efficiency**: You only pay for compute and storage you use

## 🔄 Data Flow Example

Let's see how control plane and data plane work together:

### Scenario: Running a Notebook Query

1. **User action** (Control Plane):
   * User clicks "Run" on a notebook cell in Workspace UI
   * Request goes to control plane REST API

2. **Cluster check** (Control Plane):
   * Control plane checks if cluster is running
   * If not, sends command to start cluster in data plane

3. **Code submission** (Control Plane → Data Plane):
   * Control plane sends code to cluster driver in data plane
   * Authentication and authorization checked via Unity Catalog

4. **Query execution** (Data Plane):
   * Spark driver creates execution plan
   * Tasks distributed to worker nodes
   * Data read from S3/ADLS/GCS
   * Computation happens on cluster

5. **Results collection** (Data Plane → Control Plane):
   * First 1000 rows sent back to control plane
   * Results displayed in notebook UI
   * Full results remain in data plane

6. **Monitoring** (Both planes):
   * Control plane logs the job execution
   * Data plane metrics sent to control plane for display
   * Unity Catalog records lineage and access logs

In [0]:
# Check which cloud provider and region your data plane is in
import requests
import json

# Get cluster information
try:
    # Get current cluster details
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    
    print("=" * 60)
    print("DATA PLANE INFORMATION")
    print("=" * 60)
    print(f"\nWorkspace URL: {workspace_url}")
    print(f"Cluster ID: {cluster_id}")
    
    # Get cloud provider from tags
    cloud_provider = spark.conf.get("spark.databricks.cloudProvider", "Unknown")
    print(f"\nCloud Provider: {cloud_provider}")
    
    # Get region information
    try:
        region = spark.conf.get("spark.databricks.clusterUsageTags.region", "Unknown")
        print(f"Region: {region}")
    except:
        print("Region: Unknown")
    
    print("\n" + "=" * 60)
    print("CONTROL PLANE")
    print("=" * 60)
    print("\nLocation: Databricks-managed cloud account")
    print("Access via: https://" + workspace_url)
    
    print("\n" + "=" * 60)
    print("\n✅ Your data is processed in YOUR cloud account (Data Plane)")
    print("✅ Your workspace UI is served from Databricks cloud (Control Plane)")
    
except Exception as e:
    print(f"Unable to retrieve cluster information: {e}")
    print("\nThis is normal if running on serverless compute.")

## 🎓 Summary and Best Practices

### Unity Catalog Best Practices

1. **Organize by environment**: Use separate catalogs for `dev`, `staging`, `prod`
2. **Schema naming**: Group related tables in schemas by domain (e.g., `sales`, `marketing`, `finance`)
3. **Document everything**: Add comments to catalogs, schemas, tables, and columns
4. **Enable CDF early**: Turn on Change Data Feed for tables that need audit trails
5. **Use qualified names**: Always use `catalog.schema.table` to avoid ambiguity

### Access Control Best Practices

1. **Principle of least privilege**: Grant only necessary permissions
2. **Use groups, not individuals**: Manage permissions via groups for easier maintenance
3. **Regular audits**: Review SHOW GRANTS periodically
4. **Protect PII**: Apply column masks and row filters for sensitive data
5. **Document access patterns**: Comment on why specific permissions were granted
6. **Separate duties**: Different groups for read (analysts) vs write (engineers)

### CDC Best Practices

1. **Choose the right approach**: 
   * Simple updates: Use MERGE
   * Complex workflows: Use Lakeflow Pipelines
   * Real-time: Use Auto Loader with CDC

2. **Enable CDF on source tables**: Track all changes automatically
3. **Use sequence columns**: Always have a way to order changes (timestamp, version)
4. **Handle late arrivals**: Design for out-of-order data
5. **Test with deletes**: Ensure your CDC handles all operation types

### Architecture Best Practices

1. **Understand separation**: Know what runs where (control vs data plane)
2. **Secure your data plane**: Use VPC, private links, encryption
3. **Monitor both planes**: Watch metrics from both control and data plane
4. **Cost optimization**: Terminate idle clusters (data plane costs)
5. **Network planning**: Ensure data plane can reach your data sources

---

## 🎉 Congratulations!

You now understand:
* ✅ Unity Catalog hierarchy and governance
* ✅ Fine-grained access control with PII protection
* ✅ CDC patterns for incremental data processing
* ✅ Databricks control plane vs data plane architecture

**Next steps**: Run the code cells to see these concepts in action!

## 🥉🥈🥇 Medallion Architecture Overview

**Medallion Architecture** is a data design pattern used to organize data in a lakehouse incrementally and progressively as it flows through layers.

### The Three Layers

```
┌─────────────────────────────────────────────────────────┐
│  🥉 BRONZE LAYER (Raw/Landing)                         │
│  • Raw ingested data (minimal processing)              │
│  • Preserve original format and history                │
│  • Append-only, immutable                              │
│  • Source of truth                                     │
└─────────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────┐
│  🥈 SILVER LAYER (Refined/Cleansed)                    │
│  • Validated, cleansed, enriched data                  │
│  • Standardized schema                                 │
│  • Deduplicated records                                │
│  • Data quality rules applied                          │
│  • Business keys and joins added                       │
└─────────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────┐
│  🥇 GOLD LAYER (Business/Aggregated)                   │
│  • Business-level aggregates                           │
│  • Feature tables for ML                               │
│  • Reporting tables                                    │
│  • Highly performant, optimized                        │
│  • Domain-specific models                              │
└─────────────────────────────────────────────────────────┘
```

### Why Medallion Architecture?

* **Separation of concerns**: Each layer has clear responsibilities
* **Incremental refinement**: Data quality improves through layers
* **Flexibility**: Can rebuild downstream from any layer
* **Auditability**: Track data lineage and transformations
* **Performance**: Optimize each layer for its use case

## 🥈 Silver Layer: The Refined Data Layer

**The Silver Layer** is where raw data becomes useful for analytics and downstream processing.

### Key Characteristics

* **Cleansed**: Invalid records removed or fixed
* **Validated**: Data quality rules enforced
* **Standardized**: Consistent schemas and formats
* **Deduplicated**: No duplicate records
* **Enriched**: Lookups and joins added
* **Typed**: Proper data types (not all strings)
* **Partitioned**: Optimized for query performance

### Common Transformations

1. **Data Type Conversion**
   * String → Date/Timestamp
   * String → Integer/Decimal
   * JSON strings → Structured columns

2. **Data Cleansing**
   * Remove null/invalid values
   * Trim whitespace
   * Standardize formats (phone, email, addresses)
   * Fix encoding issues

3. **Deduplication**
   * Remove exact duplicates
   * Keep latest record by timestamp
   * Merge duplicate records with business logic

4. **Enrichment**
   * Add reference data (country codes, categories)
   * Join with dimension tables
   * Calculate derived fields
   * Add metadata columns (processing_time, source_file)

5. **Data Quality Checks**
   * NOT NULL constraints
   * Range checks (dates, amounts)
   * Format validation (email, phone)
   * Referential integrity

### Silver Layer Schema Design

* Use meaningful column names
* Add column comments for documentation
* Include audit columns: `_processing_time`, `_source_file`
* Partition by date for performance
* Enable Change Data Feed for downstream tracking

## 🔧 End-to-End Silver Layer Pipeline

Let's build a complete Bronze → Silver pipeline processing e-commerce order data.

In [0]:
%sql
INSERT INTO bronze_orders VALUES
  ('ORD001', 'CUST001', '2024-05-01', '150.50', 'completed', 'credit_card', 'john@email.com', '123 Main St, Seattle, WA', 'P001,P002', current_timestamp(), 's3://bucket/orders/file1.json', current_timestamp()),
  ('ORD002', 'CUST002', '2024-05-01', '275.00', 'pending', 'paypal', 'jane@email.com', '456 Oak Ave, Portland, OR', 'P003', current_timestamp(), 's3://bucket/orders/file1.json', current_timestamp()),
  ('ORD003', 'CUST001', '2024-05-02', '89.99', 'completed', 'credit_card', 'john@email.com', '123 Main St, Seattle, WA', 'P004,P005,P006', current_timestamp(), 's3://bucket/orders/file2.json', current_timestamp()),
  ('ORD004', 'CUST003', 'invalid-date', 'not-a-number', 'shipped', 'credit_card', 'invalid-email', '789 Pine Rd, SF, CA', 'P001', current_timestamp(), 's3://bucket/orders/file2.json', current_timestamp()),
  ('ORD001', 'CUST001', '2024-05-01', '150.50', 'completed', 'credit_card', 'john@email.com', '123 Main St, Seattle, WA', 'P001,P002', current_timestamp(), 's3://bucket/orders/file3.json', current_timestamp()); -- Duplicate

In [0]:
%sql
-- Create a schema for our medallion layers
CREATE SCHEMA IF NOT EXISTS my_catalog.ecommerce
COMMENT 'E-commerce data processing layers';

USE CATALOG my_catalog;
USE SCHEMA ecommerce;

-- Bronze layer: Raw ingested data
CREATE OR REPLACE TABLE bronze_orders (
  order_id STRING,
  customer_id STRING,
  order_date STRING,           -- Raw string from source
  order_amount STRING,         -- Raw string from source
  order_status STRING,
  payment_method STRING,
  customer_email STRING,
  shipping_address STRING,
  product_ids STRING,          -- Comma-separated list
  created_at TIMESTAMP,
  _source_file STRING,
  _ingestion_time TIMESTAMP
)
COMMENT 'Bronze layer: Raw order data as ingested';

-- Insert sample raw data (simulating data ingestion)
INSERT INTO bronze_orders VALUES
  ('ORD001', 'CUST001', '2024-05-01', '150.50', 'completed', 'credit_card', 'john@email.com', '123 Main St, Seattle, WA', 'P001,P002', current_timestamp(), 's3://bucket/orders/file1.json', current_timestamp()),
  ('ORD002', 'CUST002', '2024-05-01', '275.00', 'pending', 'paypal', 'jane@email.com', '456 Oak Ave, Portland, OR', 'P003', current_timestamp(), 's3://bucket/orders/file1.json', current_timestamp()),
  ('ORD003', 'CUST001', '2024-05-02', '89.99', 'completed', 'credit_card', 'john@email.com', '123 Main St, Seattle, WA', 'P004,P005,P006', current_timestamp(), 's3://bucket/orders/file2.json', current_timestamp()),
  ('ORD004', 'CUST003', 'invalid-date', 'not-a-number', 'shipped', 'credit_card', 'invalid-email', '789 Pine Rd, SF, CA', 'P001', current_timestamp(), 's3://bucket/orders/file2.json', current_timestamp()),
  ('ORD001', 'CUST001', '2024-05-01', '150.50', 'completed', 'credit_card', 'john@email.com', '123 Main St, Seattle, WA', 'P001,P002', current_timestamp(), 's3://bucket/orders/file3.json', current_timestamp()); -- Duplicate

SELECT * FROM bronze_orders;

In [0]:
from pyspark.sql.functions import (
    col, to_date, to_timestamp, regexp_replace, trim, lower, upper,
    split, size, array_distinct, current_timestamp, when, 
    row_number, coalesce, regexp_extract, expr
)
from pyspark.sql.window import Window

# Read from Bronze layer
bronze_df = spark.table("my_catalog.ecommerce.bronze_orders")

print(f"Bronze records: {bronze_df.count()}")

# Step 1: Data Type Conversions
silver_df = bronze_df \
    .withColumn("order_id", trim(col("order_id"))) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn(
        "order_date_converted", 
        expr("try_to_date(order_date, 'yyyy-MM-dd')")
    ) \
    .withColumn(
        "order_amount_converted", 
        expr("try_cast(order_amount as decimal(10,2))")
    )
print(f"Data Type Conversions: {display(silver_df)}")


# Step 2: Data Cleansing
silver_df = silver_df \
    .withColumn("order_status", lower(trim(col("order_status")))) \
    .withColumn("payment_method", lower(trim(col("payment_method")))) \
    .withColumn("customer_email", lower(trim(col("customer_email")))) \
    .withColumn("shipping_address", trim(col("shipping_address")))
display(silver_df)
# Step 3: Data Quality Filtering
# Mark invalid records
silver_df = silver_df \
    .withColumn(
        "is_valid",
        when(
            (col("order_date_converted").isNull()) |
            (col("order_amount_converted").isNull()) |
            (col("order_amount_converted") <= 0) |
            (~col("customer_email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")),
            False
        ).otherwise(True)
    )
display(silver_df)
print(f"Valid records: {silver_df.filter(col('is_valid') == True).count()}")
print(f"Invalid records: {silver_df.filter(col('is_valid') == False).count()}")

# Step 4: Deduplication
# Keep the latest record for each order_id
window_spec = Window.partitionBy("order_id").orderBy(col("_ingestion_time").desc())

silver_df = silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1)
display(silver_df)

silver_df = silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

print(f"After deduplication: {silver_df.count()}")
display(silver_df)

# Step 5: Enrichment
# Split product IDs into array and count
silver_df = silver_df \
    .withColumn("product_ids_array", split(col("product_ids"), ",")) \
    .withColumn("num_products", size(col("product_ids_array"))) \
    .withColumn("_processing_time", current_timestamp())
display(silver_df)
# Display sample
print("\nSample Silver Layer Records:")
display(silver_df.select(
    "order_id", "customer_id", "order_date_converted", 
    "order_amount_converted", "order_status", "num_products", "is_valid"
).orderBy("order_id"))

In [0]:
# Filter only valid records for Silver layer
valid_silver_df = silver_df.filter(col("is_valid") == True)

# Select final schema for Silver layer
final_silver_df = valid_silver_df.select(
    col("order_id"),
    col("customer_id"),
    col("order_date_converted").alias("order_date"),
    col("order_amount_converted").alias("order_amount"),
    col("order_status"),
    col("payment_method"),
    col("customer_email"),
    col("shipping_address"),
    col("product_ids_array").alias("product_ids"),
    col("num_products"),
    col("_source_file"),
    col("_ingestion_time"),
    col("_processing_time")
)

# Write to Silver layer table (partitioned by order date)
final_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_date") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("my_catalog.ecommerce.silver_orders")

print("✅ Silver layer table created successfully!")
print(f"Total valid records written: {final_silver_df.count()}")

# Optimize the table
spark.sql("OPTIMIZE my_catalog.ecommerce.silver_orders")
print("✅ Table optimized")

# Display final Silver layer
print("\nFinal Silver Layer Table:")
display(spark.table("my_catalog.ecommerce.silver_orders"))

In [0]:
%sql
-- Add table comment
COMMENT ON TABLE my_catalog.ecommerce.silver_orders IS 
  'Silver layer: Validated, cleansed, and enriched order data';

-- Add column comments for documentation
ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN order_id COMMENT 'Unique order identifier';

ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN customer_id COMMENT 'Customer identifier';

ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN order_date COMMENT 'Date when order was placed';

ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN order_amount COMMENT 'Total order amount in USD';

ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN product_ids COMMENT 'Array of product IDs in the order';

ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN num_products COMMENT 'Number of products in the order';

ALTER TABLE my_catalog.ecommerce.silver_orders 
  ALTER COLUMN _processing_time COMMENT 'Timestamp when record was processed to Silver';

-- Verify table properties
DESCRIBE EXTENDED my_catalog.ecommerce.silver_orders;

In [0]:
# Good practice: Store invalid records in a quarantine table for investigation
invalid_records_df = silver_df.filter(col("is_valid") == False)

if invalid_records_df.count() > 0:
    # Add reason for invalidity
    invalid_with_reason = invalid_records_df.withColumn(
        "invalid_reason",
        when(col("order_date_converted").isNull(), "Invalid order date")
        .when(col("order_amount_converted").isNull(), "Invalid order amount")
        .when(col("order_amount_converted") <= 0, "Order amount must be positive")
        .when(~col("customer_email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"), 
              "Invalid email format")
        .otherwise("Unknown reason")
    )
    
    # Write to quarantine table
    invalid_with_reason.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("my_catalog.ecommerce.quarantine_orders")
    
    print(f"⚠️  {invalid_records_df.count()} invalid records written to quarantine table")
    
    # Display invalid records
    print("\nInvalid Records for Investigation:")
    display(spark.table("my_catalog.ecommerce.quarantine_orders").select(
        "order_id", "order_date", "order_amount", "customer_email", "invalid_reason"
    ))
else:
    print("✅ No invalid records found")

In [0]:
# Now let's query the Silver layer for analytics

print("=" * 60)
print("SILVER LAYER ANALYTICS")
print("=" * 60)

# Query 1: Orders by status
orders_by_status = spark.sql("""
    SELECT 
        order_status,
        COUNT(*) as order_count,
        SUM(order_amount) as total_revenue,
        AVG(order_amount) as avg_order_value,
        AVG(num_products) as avg_products_per_order
    FROM my_catalog.ecommerce.silver_orders
    GROUP BY order_status
    ORDER BY order_count DESC
""")

print("\n1. Orders by Status:")
display(orders_by_status)

# Query 2: Daily order summary
daily_summary = spark.sql("""
    SELECT 
        order_date,
        COUNT(*) as num_orders,
        SUM(order_amount) as daily_revenue,
        COUNT(DISTINCT customer_id) as unique_customers
    FROM my_catalog.ecommerce.silver_orders
    GROUP BY order_date
    ORDER BY order_date DESC
""")

print("\n2. Daily Order Summary:")
display(daily_summary)

# Query 3: Payment method distribution
payment_methods = spark.sql("""
    SELECT 
        payment_method,
        COUNT(*) as order_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM my_catalog.ecommerce.silver_orders
    GROUP BY payment_method
    ORDER BY order_count DESC
""")

print("\n3. Payment Method Distribution:")
display(payment_methods)

## ✅ Silver Layer Data Quality Framework

### Data Quality Dimensions

1. **Completeness**
   * Are all expected records present?
   * Are required fields populated?

2. **Accuracy**
   * Are values correct and within valid ranges?
   * Do calculations produce expected results?

3. **Consistency**
   * Do related fields agree (e.g., order_date ≤ ship_date)?
   * Are formats standardized?

4. **Timeliness**
   * Is data fresh enough for downstream use?
   * Are there unexpected delays?

5. **Uniqueness**
   * Are there duplicate records?
   * Are primary keys unique?

### Implementing Data Quality Checks

In [0]:
from pyspark.sql.functions import count, countDistinct, sum as _sum, avg, min as _min, max as _max

print("=" * 60)
print("DATA QUALITY REPORT - SILVER LAYER")
print("=" * 60)

silver_table = spark.table("my_catalog.ecommerce.silver_orders")

# 1. Completeness Checks
print("\n1️⃣  COMPLETENESS CHECKS")
print("-" * 60)

total_records = silver_table.count()
print(f"Total records: {total_records}")

for column in ["order_id", "customer_id", "order_date", "order_amount"]:
    null_count = silver_table.filter(col(column).isNull()).count()
    null_pct = (null_count / total_records * 100) if total_records > 0 else 0
    status = "✅" if null_count == 0 else "⚠️"
    print(f"{status} {column}: {null_count} nulls ({null_pct:.2f}%)")

# 2. Accuracy Checks
print("\n2️⃣  ACCURACY CHECKS")
print("-" * 60)

# Check for negative amounts
negative_amounts = silver_table.filter(col("order_amount") < 0).count()
print(f"{'✅' if negative_amounts == 0 else '⚠️'} Negative order amounts: {negative_amounts}")

# Check for future dates
future_dates = silver_table.filter(col("order_date") > current_timestamp()).count()
print(f"{'✅' if future_dates == 0 else '⚠️'} Future order dates: {future_dates}")

# Check email format
invalid_emails = silver_table.filter(
    ~col("customer_email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
).count()
print(f"{'✅' if invalid_emails == 0 else '⚠️'} Invalid email formats: {invalid_emails}")

# 3. Uniqueness Checks
print("\n3️⃣  UNIQUENESS CHECKS")
print("-" * 60)

duplicates = total_records - silver_table.select("order_id").distinct().count()
print(f"{'✅' if duplicates == 0 else '⚠️'} Duplicate order_ids: {duplicates}")

# 4. Statistical Summary
print("\n4️⃣  STATISTICAL SUMMARY")
print("-" * 60)

stats = silver_table.agg(
    _min("order_amount").alias("min_amount"),
    _max("order_amount").alias("max_amount"),
    avg("order_amount").alias("avg_amount"),
    _min("order_date").alias("earliest_date"),
    _max("order_date").alias("latest_date")
).collect()[0]

print(f"Order Amount Range: ${stats['min_amount']:.2f} - ${stats['max_amount']:.2f}")
print(f"Average Order Amount: ${stats['avg_amount']:.2f}")
print(f"Date Range: {stats['earliest_date']} to {stats['latest_date']}")

# 5. Freshness Check
print("\n5️⃣  DATA FRESHNESS")
print("-" * 60)

latest_processing = silver_table.agg(_max("_processing_time")).collect()[0][0]
print(f"Latest processing time: {latest_processing}")

print("\n" + "=" * 60)
print("DATA QUALITY REPORT COMPLETE")
print("=" * 60)

## 📋 Silver Layer Pipeline Best Practices

### 1. Schema Management

* **Version your schemas**: Document schema changes over time
* **Use explicit schemas**: Don't rely on schema inference in production
* **Add column comments**: Document business meaning of each field
* **Use appropriate data types**: Don't store everything as strings

### 2. Performance Optimization

* **Partition strategically**: Use date partitioning for time-series data
* **Z-order optimization**: Cluster by commonly filtered columns
* **Optimize file sizes**: Run OPTIMIZE regularly (target: 128MB-1GB files)
* **Enable Auto Compaction**: For streaming writes
* **Use Delta Lake liquid clustering**: For high-cardinality partitioning columns

### 3. Data Quality

* **Implement validation rules**: Check data quality at ingestion
* **Create quarantine tables**: Store invalid records for investigation
* **Monitor data quality metrics**: Track completeness, accuracy, timeliness
* **Set up alerts**: Notify when quality thresholds are breached
* **Document quality rules**: Make expectations explicit

### 4. Incremental Processing

* **Use watermarks**: For streaming data
* **Implement checkpoints**: For resumable processing
* **Track processed files**: Avoid reprocessing
* **Handle late-arriving data**: Design for out-of-order records
* **Use MERGE for upserts**: Maintain current state efficiently

### 5. Monitoring and Observability

* **Log pipeline runs**: Track start time, duration, records processed
* **Track data lineage**: Know where data came from and where it goes
* **Monitor SLAs**: Ensure data freshness requirements are met
* **Set up dashboards**: Visualize pipeline health
* **Enable Change Data Feed**: Track all table changes

### 6. Error Handling

* **Graceful degradation**: Don't fail entire batch for single bad record
* **Retry logic**: Handle transient failures
* **Dead letter queues**: Store unprocessable records
* **Logging**: Capture detailed error context
* **Alerting**: Notify on-call team of critical failures

### 7. Testing

* **Unit tests**: Test transformation logic
* **Integration tests**: Test end-to-end pipeline
* **Data validation tests**: Verify quality rules
* **Regression tests**: Ensure changes don't break existing logic
* **Load tests**: Validate performance at scale

### 8. Code Organization

```python
# Recommended structure
def read_bronze(table_name):
    """Read from Bronze layer"""
    return spark.table(table_name)

def apply_transformations(df):
    """Apply all Silver layer transformations"""
    df = convert_data_types(df)
    df = cleanse_data(df)
    df = deduplicate(df)
    df = enrich_data(df)
    return df

def validate_data_quality(df):
    """Run data quality checks"""
    return df.filter(col("is_valid") == True)

def write_silver(df, table_name):
    """Write to Silver layer"""
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

# Main pipeline
bronze_df = read_bronze("bronze_orders")
silver_df = apply_transformations(bronze_df)
valid_df = validate_data_quality(silver_df)
write_silver(valid_df, "silver_orders")
```

---

## 🎯 Summary

The Silver Layer is the **backbone of your data lakehouse**:

* ✅ Transforms raw data into clean, validated datasets
* ✅ Enforces data quality and business rules
* ✅ Provides reliable foundation for Gold layer and analytics
* ✅ Enables self-service analytics with trusted data
* ✅ Tracks lineage and supports compliance requirements

**Next**: Build Gold layer aggregates and feature tables on top of your Silver foundation!

## 🥇 Gold Layer: Business-Ready Data

**The Gold Layer** contains highly refined, aggregated data optimized for specific business use cases.

### Key Characteristics

* **Business-level aggregations**: Pre-calculated metrics and KPIs
* **Denormalized**: Optimized for query performance
* **Use-case specific**: Tailored to analytics, dashboards, ML features
* **Highly performant**: Small, fast tables with minimal joins
* **Schema stable**: Rarely changes, maintains backward compatibility

### Common Gold Layer Tables

1. **Customer Metrics**
   * Lifetime value, order frequency, recency
   * Customer segments and cohorts
   * Churn predictions

2. **Product Metrics**
   * Revenue by product, category
   * Inventory turnover
   * Product recommendations

3. **Time-Series Aggregates**
   * Daily/weekly/monthly summaries
   * Year-over-year comparisons
   * Trend analysis

4. **KPI Dashboards**
   * Executive summaries
   * Operational metrics
   * Real-time monitoring views

5. **ML Feature Tables**
   * Pre-computed features for models
   * Point-in-time correct features
   * Feature store integration

---

Let's build Gold layer tables from our Silver orders data!

In [0]:
# Gold Layer Table 1: Customer Metrics
# Aggregate customer-level statistics for business analysis

from pyspark.sql.functions import (
    col, count, sum as _sum, avg, min as _min, max as _max,
    datediff, current_date, first, countDistinct
)

print("Building Gold Layer: Customer Metrics")
print("=" * 60)

# Read from Silver layer
silver_orders = spark.table("my_catalog.ecommerce.silver_orders")

# Calculate customer-level metrics
customer_metrics = silver_orders.groupBy("customer_id").agg(
    # Order metrics
    count("order_id").alias("total_orders"),
    countDistinct("order_id").alias("unique_orders"),
    
    # Revenue metrics
    _sum("order_amount").alias("lifetime_value"),
    avg("order_amount").alias("avg_order_value"),
    _min("order_amount").alias("min_order_value"),
    _max("order_amount").alias("max_order_value"),
    
    # Product metrics
    _sum("num_products").alias("total_products_purchased"),
    avg("num_products").alias("avg_products_per_order"),
    
    # Recency metrics
    _min("order_date").alias("first_order_date"),
    _max("order_date").alias("last_order_date"),
    
    # Payment preferences
    first("payment_method").alias("most_recent_payment_method"),
    first("customer_email").alias("email")
)

# Add derived metrics
customer_metrics = customer_metrics \
    .withColumn(
        "days_since_first_order",
        datediff(current_date(), col("first_order_date"))
    ) \
    .withColumn(
        "days_since_last_order",
        datediff(current_date(), col("last_order_date"))
    ) \
    .withColumn(
        "customer_tenure_days",
        datediff(col("last_order_date"), col("first_order_date"))
    )

# Write to Gold layer
customer_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.ecommerce.gold_customer_metrics")

print(f"✅ Created gold_customer_metrics table with {customer_metrics.count()} customers")

# Display sample
print("\nSample Customer Metrics:")
display(customer_metrics.orderBy(col("lifetime_value").desc()))

In [0]:
# Gold Layer Table 2: Product Performance Metrics
# Analyze product popularity and revenue contribution

from pyspark.sql.functions import explode, collect_set, array_sort

print("Building Gold Layer: Product Metrics")
print("=" * 60)

# Read from Silver layer
silver_orders = spark.table("my_catalog.ecommerce.silver_orders")

# Explode product arrays to get one row per product per order
product_orders = silver_orders.select(
    col("order_id"),
    col("customer_id"),
    col("order_date"),
    col("order_amount"),
    col("order_status"),
    explode(col("product_ids")).alias("product_id")
)

# Calculate product-level metrics
product_metrics = product_orders.groupBy("product_id").agg(
    # Order metrics
    count("order_id").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    
    # Revenue metrics (evenly split order amount across products)
    _sum("order_amount").alias("total_revenue"),
    avg("order_amount").alias("avg_order_value"),
    
    # Date metrics
    _min("order_date").alias("first_order_date"),
    _max("order_date").alias("last_order_date"),
    
    # Status distribution
    _sum(when(col("order_status") == "completed", 1).otherwise(0)).alias("completed_orders"),
    _sum(when(col("order_status") == "pending", 1).otherwise(0)).alias("pending_orders")
)

# Add derived metrics
product_metrics = product_metrics \
    .withColumn(
        "completion_rate",
        (col("completed_orders") / col("total_orders") * 100)
    ) \
    .withColumn(
        "revenue_per_customer",
        col("total_revenue") / col("unique_customers")
    )

# Write to Gold layer
product_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.ecommerce.gold_product_metrics")

print(f"✅ Created gold_product_metrics table with {product_metrics.count()} products")

# Display sample
print("\nTop Products by Revenue:")
display(product_metrics.orderBy(col("total_revenue").desc()))

In [0]:
# Gold Layer Table 3: Daily Business Metrics
# Time-series data for dashboards and trend analysis

from pyspark.sql.functions import date_format, dayofweek

print("Building Gold Layer: Daily Metrics")
print("=" * 60)

# Read from Silver layer
silver_orders = spark.table("my_catalog.ecommerce.silver_orders")

# Calculate daily aggregates
daily_metrics = silver_orders.groupBy("order_date").agg(
    # Order metrics
    count("order_id").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    
    # Revenue metrics
    _sum("order_amount").alias("total_revenue"),
    avg("order_amount").alias("avg_order_value"),
    _min("order_amount").alias("min_order_value"),
    _max("order_amount").alias("max_order_value"),
    
    # Product metrics
    _sum("num_products").alias("total_products_sold"),
    avg("num_products").alias("avg_products_per_order"),
    
    # Status breakdown
    _sum(when(col("order_status") == "completed", 1).otherwise(0)).alias("completed_orders"),
    _sum(when(col("order_status") == "pending", 1).otherwise(0)).alias("pending_orders"),
    _sum(when(col("order_status") == "shipped", 1).otherwise(0)).alias("shipped_orders"),
    
    # Payment method breakdown
    _sum(when(col("payment_method") == "credit_card", 1).otherwise(0)).alias("credit_card_orders"),
    _sum(when(col("payment_method") == "paypal", 1).otherwise(0)).alias("paypal_orders")
)

# Add derived metrics and date attributes
daily_metrics = daily_metrics \
    .withColumn("day_of_week", dayofweek(col("order_date"))) \
    .withColumn("day_name", date_format(col("order_date"), "EEEE")) \
    .withColumn(
        "revenue_per_customer",
        col("total_revenue") / col("unique_customers")
    ) \
    .withColumn(
        "completion_rate_pct",
        (col("completed_orders") / col("total_orders") * 100)
    ) \
    .orderBy("order_date")

# Write to Gold layer
daily_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.ecommerce.gold_daily_metrics")

print(f"✅ Created gold_daily_metrics table with {daily_metrics.count()} days")

# Display sample
print("\nDaily Business Metrics:")
display(daily_metrics)

In [0]:
# Gold Layer Table 4: Executive KPI Dashboard
# Single-row summary for high-level business monitoring

from pyspark.sql.functions import lit, current_timestamp

print("Building Gold Layer: Executive KPI Dashboard")
print("=" * 60)

# Read from Silver layer
silver_orders = spark.table("my_catalog.ecommerce.silver_orders")

# Calculate business KPIs
kpi_dashboard = silver_orders.agg(
    # Order KPIs
    count("order_id").alias("total_orders"),
    countDistinct("customer_id").alias("total_customers"),
    
    # Revenue KPIs
    _sum("order_amount").alias("total_revenue"),
    avg("order_amount").alias("avg_order_value"),
    
    # Product KPIs
    _sum("num_products").alias("total_products_sold"),
    avg("num_products").alias("avg_products_per_order"),
    
    # Date range
    _min("order_date").alias("earliest_order_date"),
    _max("order_date").alias("latest_order_date")
)

# Add derived KPIs
kpi_dashboard = kpi_dashboard \
    .withColumn(
        "revenue_per_customer",
        col("total_revenue") / col("total_customers")
    ) \
    .withColumn(
        "orders_per_customer",
        col("total_orders") / col("total_customers")
    ) \
    .withColumn(
        "reporting_period_days",
        datediff(col("latest_order_date"), col("earliest_order_date")) + 1
    ) \
    .withColumn(
        "daily_revenue",
        col("total_revenue") / col("reporting_period_days")
    ) \
    .withColumn("snapshot_timestamp", current_timestamp())

# Write to Gold layer
kpi_dashboard.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.ecommerce.gold_executive_kpis")

print("✅ Created gold_executive_kpis dashboard table")

# Display KPIs
print("\n📊 EXECUTIVE DASHBOARD - KEY PERFORMANCE INDICATORS")
print("=" * 60)
display(kpi_dashboard)

# Pretty print key metrics
kpis = kpi_dashboard.collect()[0]
print("\n💰 REVENUE METRICS")
print(f"  Total Revenue: ${kpis['total_revenue']:,.2f}")
print(f"  Avg Order Value: ${kpis['avg_order_value']:,.2f}")
print(f"  Revenue per Customer: ${kpis['revenue_per_customer']:,.2f}")
print(f"  Daily Revenue: ${kpis['daily_revenue']:,.2f}")

print("\n👥 CUSTOMER METRICS")
print(f"  Total Customers: {kpis['total_customers']:,}")
print(f"  Total Orders: {kpis['total_orders']:,}")
print(f"  Orders per Customer: {kpis['orders_per_customer']:.2f}")

print("\n📦 PRODUCT METRICS")
print(f"  Total Products Sold: {kpis['total_products_sold']:,}")
print(f"  Avg Products per Order: {kpis['avg_products_per_order']:.2f}")

In [0]:
%sql
-- Add documentation to Gold layer tables

COMMENT ON TABLE my_catalog.ecommerce.gold_customer_metrics IS
  'Gold layer: Customer-level aggregations including lifetime value, order frequency, and recency metrics';

COMMENT ON TABLE my_catalog.ecommerce.gold_product_metrics IS
  'Gold layer: Product-level performance metrics including revenue, order counts, and customer reach';

COMMENT ON TABLE my_catalog.ecommerce.gold_daily_metrics IS
  'Gold layer: Daily business metrics for trend analysis and time-series dashboards';

COMMENT ON TABLE my_catalog.ecommerce.gold_executive_kpis IS
  'Gold layer: Executive dashboard with high-level KPIs and business health indicators';

-- Verify Gold layer tables
SHOW TABLES IN my_catalog.ecommerce LIKE 'gold*';

## 🎯 Gold Layer Complete!

### What We Built

We created a complete **Bronze → Silver → Gold** medallion architecture:

**🥉 Bronze Layer** (`bronze_orders`)
* Raw ingested data
* 5 records with intentional data quality issues

**🥈 Silver Layer** (`silver_orders`)
* Validated, cleansed data
* Type conversions (dates, decimals)
* Data quality filtering
* Deduplication
* Enrichment (product arrays, counts)
* 4 valid records written

**🥇 Gold Layer** (4 tables)
1. `gold_customer_metrics` - Customer lifetime value, order frequency
2. `gold_product_metrics` - Product revenue and popularity
3. `gold_daily_metrics` - Time-series for dashboards
4. `gold_executive_kpis` - Single-row KPI summary

---

### Gold Layer Design Principles

✅ **Pre-aggregate everything** - No complex queries at read time

✅ **Denormalize for performance** - Accept data duplication for speed

✅ **Schema stability** - Gold tables change rarely, maintain backward compatibility

✅ **Business-friendly naming** - Use terms stakeholders understand

✅ **Optimized for use case** - Each table serves specific analytics needs

✅ **Small and fast** - Gold tables should be orders of magnitude smaller than Silver

---

### Refresh Strategy

Gold layer tables are typically refreshed:

* **Batch refresh**: Daily/hourly scheduled jobs
* **Incremental refresh**: Process only new Silver records
* **Streaming**: Real-time updates for operational dashboards
* **On-demand**: Triggered by upstream Silver updates

```python
# Example: Incremental refresh using watermarks
gold_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_table")
```

---

### Use Cases

**📊 BI Dashboards**
* Connect Tableau/PowerBI directly to Gold tables
* Fast query response times
* No complex SQL needed

**🤖 ML Feature Store**
* Pre-computed features for models
* Point-in-time correctness
* Consistent feature definitions

**📈 Executive Reporting**
* Single-page KPI summaries
* Trend analysis
* Automated email reports

**🔍 Self-Service Analytics**
* Business users can query directly
* No SQL expertise required
* Governed, trusted metrics